# Scene Weaver — GPU encoder (Google Colab T4)

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Copy the **https link** printed at the bottom and paste it into the app's *Colab GPU encoder* box.

Keep this tab open while the video renders.

In [ ]:
#@title 1 · Check GPU + install tools
!nvidia-smi -L || echo 'NO GPU — set Runtime type to T4 GPU'
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -q h264_nvenc && echo 'NVENC hardware encoder: available' || echo 'NVENC missing — will fall back to CPU'

In [ ]:
#@title 2 · Start the encoder + public https tunnel
APP_URL = "https://script-to-epic.lovable.app"  #@param {type:"string"}
TOKEN   = ""  #@param {type:"string"}

import os, re, subprocess, threading, time, urllib.request
os.environ['SW_TOKEN'] = TOKEN
urllib.request.urlretrieve(APP_URL.rstrip('/') + '/colab/encoder_server.py', '/content/encoder_server.py')

import importlib.util
spec = importlib.util.spec_from_file_location('encoder_server', '/content/encoder_server.py')
enc = importlib.util.module_from_spec(spec); spec.loader.exec_module(enc)
threading.Thread(target=enc.serve, kwargs={'port': 8000}, daemon=True).start()
time.sleep(2)
print('encoder running · GPU NVENC =', enc.NVENC)

p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in p.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '='*64)
print('PASTE THIS INTO THE APP:')
print(url or 'tunnel failed — re-run this cell')
print('='*64)

In [ ]:
#@title 3 · Keep alive (leave running while the video encodes)
import time
while True:
    time.sleep(60)